# AR(1) time series and sliding-window clouds with Beta innovations

This notebook produces three separate figures: five AR(1) time series in
stacked panels, their corresponding two-dimensional sliding-window clouds,
and an overlay of all five time series in distinct colors. All cases use the
same coefficient and innovation support $[-1,1]$; black dashed lines mark
the common theoretical support boundary in the cloud figure.

We draw
$$U_t\sim\operatorname{Beta}(\alpha,\beta),\qquad
\varepsilon_t=2U_t-1,\qquad
X_t=\phi_1X_{t-1}+\varepsilon_t,\quad |\phi_1|<1.$$
The five default shapes are $(1,1)$, $(1/2,1/2)$, $(2,2)$,
$(1/2,2)$ and $(2,1/2)$. The location-scale interpretation and Beta moments
are given in [NIST/SEMATECH, section 1.3.6.6.17, Probability Density Function
and Common Statistics](https://www.itl.nist.gov/div898/handbook/eda/section3/eda366h.htm).

For this fixed-support comparison, the transformed moments are
$$\mathbb E[\varepsilon_t]=\frac{\alpha-\beta}{\alpha+\beta},\qquad
\operatorname{Var}(\varepsilon_t)=
\frac{4\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}.$$
These follow directly from the cited Beta moments. The notebook does not
recenter or standardize the innovations. In particular, asymmetric shapes
have nonzero innovation means. Each AR(1) run starts at its theoretical
mean $\mathbb E[\varepsilon_t]/(1-\phi_1)$ and discards the configured burn-in;
this mean follows by taking expectations in the displayed AR(1) equation.

The stacked time-series panels show the first `time_series_points`
observations of each path. The clouds use all `sample_size - 1` consecutive
pairs from those same paths. The overlay shows the first `overlay_points`
observations, using a shorter window to keep five colored curves readable.
Every displayed figure is saved as PDF and PNG. This figure-specific
sampler is contained in the notebook; the shared normalized AR generators
retain their existing behavior.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

AR_CODE_DIR = next(
    (
        candidate.resolve()
        for root in (Path.cwd(), *Path.cwd().parents)
        for candidate in (root / "Code" / "Code_AR_processes", root)
        if (candidate / "ar_tools/functions.py").is_file()
        and (candidate / "ar_tools/support_geometry.py").is_file()
    ),
    None,
)
if AR_CODE_DIR is None:
    raise FileNotFoundError("Run this notebook from the project or AR code folder.")
PROJECT_ROOT = AR_CODE_DIR.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Code.Code_AR_processes.ar_tools.functions import sliding_window_point_cloud

NOTEBOOK_NAME = "ar_beta_sliding_window_2d"
PDF_DIR = AR_CODE_DIR / "outputs" / "pdfs" / NOTEBOOK_NAME
IMAGE_DIR = AR_CODE_DIR / "outputs" / "images" / NOTEBOOK_NAME
PDF_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)


## Settings

Run all cells from top to bottom. The defaults use the five Beta parameter pairs
listed in the thesis. `phi[0]` is the common AR(1) coefficient; the innovation
support stays fixed at [-1,1]. The same simulated paths feed all three figures.

- `sample_size`: retained path length and number of cloud pairs plus one.
- `time_series_points`: observations shown in each separate time-series panel.
- `overlay_points`: observations shown together in the five-color overlay.

The overlay defaults to 100 observations to reduce overlap. It uses the original
values, without vertical offsets. All figures use the same color for a given
Beta shape. The seed is reset for each shape, so rerunning reproduces each path.

Each figure is exported as PDF and PNG:

- `beta_ar1_time_series`
- `beta_sliding_window_clouds_2d`
- `beta_ar1_time_series_overlay`


In [ ]:
phi = np.array([0.7])
beta_pairs = [
    (1.0, 1.0),
    (0.5, 0.5),
    (2.0, 2.0),
    (0.5, 2.0),
    (2.0, 0.5),
]

sample_size = 5_000
burnin = 1_000
process_seed = 42
time_series_points = 300
overlay_points = 100  # Shorter window for the five colored curves.
point_size = 4
point_alpha = 0.35


In [ ]:
from Code.Code_AR_processes.ar_tools.support_geometry import theoretical_support_zonoid_2d


In [ ]:
phi = np.asarray(phi, dtype=float)
if phi.shape != (1,) or not np.all(np.isfinite(phi)) or abs(phi[0]) >= 1:
    raise ValueError("This figure requires one AR(1) coefficient with |phi[0]| < 1.")
for name, value, minimum in (
    ("sample_size", sample_size, 2),
    ("burnin", burnin, 0),
    ("time_series_points", time_series_points, 2),
    ("overlay_points", overlay_points, 2),
):
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < minimum:
        raise ValueError(f"{name} must be an integer of at least {minimum}.")
shapes = np.asarray(beta_pairs, dtype=float)
if shapes.ndim != 2 or shapes.shape[1] != 2 or len(shapes) == 0 or not np.all(np.isfinite(shapes)) or np.any(shapes <= 0):
    raise ValueError("beta_pairs must contain positive finite (alpha, beta) pairs.")

# Fixed support is the figure's experimental setting for every Beta shape.
innovation_bounds = (-1.0, 1.0)
support_boundary = theoretical_support_zonoid_2d(phi, innovation_bounds)
series_list = []
innovation_samples = []
series_initial_states = []
clouds = []
zonoid_boundaries = []
panel_innovation_bounds = []
population_innovation_means = []
population_innovation_variances = []
for alpha, beta in shapes:
    rng = np.random.default_rng(process_seed)
    innovations = 2.0 * rng.beta(alpha, beta, size=burnin + sample_size) - 1.0
    innovation_mean = (alpha - beta) / (alpha + beta)
    states = np.empty(len(innovations) + 1)
    states[0] = innovation_mean / (1.0 - phi[0])
    for t, innovation in enumerate(innovations):
        states[t + 1] = phi[0] * states[t] + innovation
    series = states[burnin + 1:]
    cloud = sliding_window_point_cloud(series, embedding_dimension=2, newest_first=False)
    if not np.all(np.isfinite(cloud)):
        raise ValueError("The simulated series contains nonfinite values.")
    series_list.append(series)
    innovation_samples.append(innovations[burnin:])
    series_initial_states.append(states[burnin])
    clouds.append(cloud)
    zonoid_boundaries.append(support_boundary.copy())
    panel_innovation_bounds.append(innovation_bounds)
    population_innovation_means.append(innovation_mean)
    population_innovation_variances.append(
        4.0 * alpha * beta / ((alpha + beta)**2 * (alpha + beta + 1.0))
    )

support_lower = float(support_boundary.min())
support_upper = float(support_boundary.max())
margin = 0.045 * (support_upper - support_lower)
axis_limits = (support_lower - margin, support_upper + margin)
shown_points = min(time_series_points, sample_size)
time_indices = np.arange(1, shown_points + 1)


shown_overlay_points = min(overlay_points, sample_size)
overlay_time_indices = np.arange(1, shown_overlay_points + 1)
palette = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7"]
series_colors = [palette[index % len(palette)] for index in range(len(shapes))]
shape_labels = [fr"$\alpha={alpha:g},\ \beta={beta:g}$" for alpha, beta in shapes]
figure_title = fr"AR(1), $\phi_1={phi[0]:g}$ | Beta innovations on $[-1,1]$"

with plt.style.context("seaborn-v0_8-whitegrid"), plt.rc_context({
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "pdf.fonttype": 42,
}):
    # Separate figure: one time-series panel for each Beta shape.
    time_series_fig, time_series_axes = plt.subplots(
        len(shapes), 1, figsize=(10.4, 1.8 * len(shapes)),
        sharex=True, sharey=True, squeeze=False, constrained_layout=True,
    )
    for ax, series, color, label in zip(
        time_series_axes.flat, series_list, series_colors, shape_labels,
    ):
        ax.plot(time_indices, series[:shown_points], color=color, linewidth=0.9)
        ax.axhline(0.0, color="0.65", linewidth=0.65, linestyle=":")
        ax.set(title=label, ylabel=r"$X_t$", xlim=(1, shown_points), ylim=axis_limits)
        ax.grid(alpha=0.2)
        ax.spines[["top", "right"]].set_visible(False)
    time_series_axes[-1, 0].set_xlabel(r"Time $t$")
    time_series_fig.suptitle(
        figure_title + f"\nTime series: first {shown_points} observations", fontsize=12,
    )

    # Separate figure: the clouds from the same five retained paths.
    cloud_columns = len(shapes)
    cloud_rows = 1
    cloud_fig, cloud_axes = plt.subplots(
        cloud_rows, cloud_columns, figsize=(3.0 * cloud_columns, 3.3),
        sharex=True, sharey=True, squeeze=False, constrained_layout=True,
    )
    for ax, cloud, boundary, color, label in zip(
        cloud_axes.flat, clouds, zonoid_boundaries, series_colors, shape_labels,
    ):
        ax.scatter(cloud[:, 0], cloud[:, 1], s=point_size, alpha=point_alpha,
                   color=color, linewidths=0, rasterized=True)
        ax.plot(boundary[:, 0], boundary[:, 1], color="black", linestyle="--", linewidth=1.2)
        ax.set(title=label, xlabel=r"$X_t$",
               ylabel=r"$X_{t+1}$" if ax is cloud_axes[0, 0] else "",
               xlim=axis_limits, ylim=axis_limits)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.2)
        ax.spines[["top", "right"]].set_visible(False)
    for ax in list(cloud_axes.flat)[len(shapes):]:
        ax.set_visible(False)
    cloud_fig.suptitle(
        figure_title + f"\nSliding-window clouds: {sample_size - 1:,} pairs per path",
        fontsize=12,
    )

    # A short, unshifted overlay provides a direct five-color comparison.
    overlay_fig, overlay_ax = plt.subplots(figsize=(10.4, 4.8), constrained_layout=True)
    overlay_lines = []
    for series, color, label in zip(series_list, series_colors, shape_labels):
        line, = overlay_ax.plot(
            overlay_time_indices, series[:shown_overlay_points],
            color=color, linewidth=1.15, alpha=0.9, label=label,
        )
        overlay_lines.append(line)
    overlay_ax.axhline(0.0, color="0.65", linewidth=0.65, linestyle=":")
    overlay_ax.set(xlabel=r"Time $t$", ylabel=r"$X_t$",
                   xlim=(1, shown_overlay_points), ylim=axis_limits)
    overlay_ax.grid(alpha=0.2)
    overlay_ax.spines[["top", "right"]].set_visible(False)
    overlay_ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16),
                      ncol=min(3, len(shapes)), frameon=False, fontsize=10)
    overlay_fig.suptitle(
        figure_title + f"\nAll {len(shapes)} time series: first {shown_overlay_points} observations",
        fontsize=12,
    )

    figure_exports = {}
    for figure, figure_stem in (
        (time_series_fig, "beta_ar1_time_series"),
        (cloud_fig, "beta_sliding_window_clouds_2d"),
        (overlay_fig, "beta_ar1_time_series_overlay"),
    ):
        pdf_path = PDF_DIR / f"{figure_stem}.pdf"
        image_path = IMAGE_DIR / f"{figure_stem}.png"
        figure.savefig(pdf_path, bbox_inches="tight")
        figure.savefig(image_path, dpi=180, bbox_inches="tight")
        figure_exports[figure_stem] = {"pdf": pdf_path, "png": image_path}
        print(f"PDF: {pdf_path}")
        print(f"PNG: {image_path}")
    plt.show()
    for figure in (time_series_fig, cloud_fig, overlay_fig):
        plt.close(figure)
